## 1. ایمپورت‌ها

In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras.layers import Input, Dense, Reshape, Flatten, Dropout, LeakyReLU
from keras.layers import Conv2D, Conv2DTranspose
from keras.models import Sequential, Model, load_model
from keras.optimizers import Adam
from keras.datasets import cifar10


## 2. داده (از کش Keras لود می‌شود، دانلود مجدد انجام نمی‌شود)

In [ ]:
(X, y), (_, _) = cifar10.load_data()

# فقط یک کلاس (8 = کشتی‌ها)
X = X[y.flatten() == 8]
X = (X.astype("float32") / 127.5) - 1.
print("shape:", X.shape, "| min:", X.min(), "max:", X.max(), "mean:", X.mean())

image_shape = (32, 32, 3)
latent_dimensions = 100


## 3. معماری‌ها (بدون BatchNormalization)\nنسخه‌های قبلی در هر دو مدل BatchNorm داشتند. وقتی `discriminator.trainable = False` می‌شود، BN در مدل ترکیبی روی آمار running کار می‌کند ولی در آموزش مستقیم روی آمار خود batch؛ یعنی generator داشت discriminator متفاوتی را فریب می‌داد. معماری زیر مثل DCGAN کتاب (فصل ۱۲) فقط `LeakyReLU` و `Dropout` دارد.

In [ ]:
def build_generator():
    model = Sequential([
        Dense(8 * 8 * 128, input_dim=latent_dimensions),
        LeakyReLU(alpha=0.2),
        Reshape((8, 8, 128)),
        Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),   # 16x16
        LeakyReLU(alpha=0.2),
        Conv2DTranspose(256, kernel_size=4, strides=2, padding="same"),   # 32x32
        LeakyReLU(alpha=0.2),
        Conv2D(3, kernel_size=5, padding="same", activation="tanh"),
    ])
    noise = Input(shape=(latent_dimensions,))
    return Model(noise, model(noise), name="generator")


def build_discriminator():
    model = Sequential([
        Conv2D(64, kernel_size=4, strides=2, padding="same", input_shape=image_shape),  # 16x16
        LeakyReLU(alpha=0.2),
        Conv2D(128, kernel_size=4, strides=2, padding="same"),                           # 8x8
        LeakyReLU(alpha=0.2),
        Conv2D(128, kernel_size=4, strides=2, padding="same"),                           # 4x4
        LeakyReLU(alpha=0.2),
        Flatten(),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])
    image = Input(shape=image_shape)
    return Model(image, model(image), name="discriminator")


## 4. ساخت و کامپایل

In [ ]:
discriminator = build_discriminator()
discriminator.compile(loss="binary_crossentropy", optimizer=Adam(0.0002, 0.5))

discriminator.trainable = False

generator = build_generator()

z = Input(shape=(latent_dimensions,))
combined_network = Model(z, discriminator(generator(z)))
combined_network.compile(loss="binary_crossentropy", optimizer=Adam(0.0002, 0.5))

generator.summary()
discriminator.summary()


## 5. حلقه‌ی آموزش\n- هر ۱۰ epoch علاوه‌بر loss، دو معیار تشخیصی چاپ می‌شود:\n  - `real_acc / fake_acc`: دقت **واقعی** discriminator (جایگزین `discm_acc` که با label smoothing همیشه ۰ بود)\n  - `diversity`: میانگین انحراف معیار بین ۱۶ تصویر تولیدشده. اگر نزدیک ۰ باشد یعنی mode collapse.\n- تصاویر ۱۰تایی در یک فایل ذخیره می‌شوند.\n- چک‌پوینت قابل ادامه بعد از Restart.

In [ ]:
num_epochs = 15000
batch_size = 32
display_interval = 10
images_per_file = 10
checkpoint_interval = 100

generator_checkpoint = "generator.weights.h5"
discriminator_checkpoint = "discriminator.weights.h5"
progress_path = "gan_training_progress.json"

os.makedirs("images", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

if (os.path.exists(generator_checkpoint) and os.path.exists(discriminator_checkpoint)
        and os.path.exists(progress_path)):
    generator.load_weights(generator_checkpoint)
    discriminator.load_weights(discriminator_checkpoint)
    with open(progress_path) as f:
        start_epoch = json.load(f)["last_epoch"] + 1
    print(f"ادامه از epoch {start_epoch}")
else:
    start_epoch = 0
    print("شروع آموزش از صفر")

losses = []
image_buffer = []


def flush_image_buffer():
    if not image_buffer:
        return
    fig, axs = plt.subplots(2, 5, figsize=(12, 5))
    for idx in range(10):
        ax = axs[idx // 5, idx % 5]
        ax.axis("off")
        if idx < len(image_buffer):
            ep, img = image_buffer[idx]
            ax.imshow(img)
            ax.set_title(f"epoch {ep}", fontsize=9)
    fname = f"images/progress_{image_buffer[0][0]}_to_{image_buffer[-1][0]}.png"
    fig.savefig(fname)
    plt.close(fig)
    print(f"  -> {len(image_buffer)} نمونه در {fname} ذخیره شد")
    image_buffer.clear()


for epoch in range(start_epoch, num_epochs):
    valid_labels = 1.0 - 0.1 * np.random.random((batch_size, 1))
    fake_labels = 0.1 * np.random.random((batch_size, 1))

    # --- Discriminator ---
    real_images = X[np.random.randint(0, X.shape[0], batch_size)]
    noise = np.random.normal(0, 1, (batch_size, latent_dimensions))
    generated_images = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_images, valid_labels)
    d_loss_fake = discriminator.train_on_batch(generated_images, fake_labels)
    d_loss = 0.5 * (d_loss_real + d_loss_fake)

    # --- Generator (یک‌بار) ---
    noise = np.random.normal(0, 1, (batch_size, latent_dimensions))
    g_loss = combined_network.train_on_batch(noise, np.ones((batch_size, 1)))

    losses.append((float(d_loss), float(g_loss)))

    if epoch % display_interval == 0:
        real_acc = float(np.mean(discriminator.predict(real_images, verbose=0) > 0.5))
        fake_acc = float(np.mean(discriminator.predict(generated_images, verbose=0) < 0.5))
        div_noise = np.random.normal(0, 1, (16, latent_dimensions))
        diversity = float(generator.predict(div_noise, verbose=0).std(axis=0).mean())
        print(f"epoch {epoch}: d_loss={d_loss:.4f} g_loss={g_loss:.4f} | "
              f"real_acc={real_acc:.2f} fake_acc={fake_acc:.2f} | diversity={diversity:.3f}")

        sample = generator.predict(np.random.normal(0, 1, (1, latent_dimensions)), verbose=0)[0]
        image_buffer.append((epoch, np.clip(0.5 * sample + 0.5, 0, 1)))
        if len(image_buffer) >= images_per_file:
            flush_image_buffer()

    if epoch % checkpoint_interval == 0:
        generator.save_weights(f"checkpoints/generator_epoch{epoch}.weights.h5")
        discriminator.save_weights(f"checkpoints/discriminator_epoch{epoch}.weights.h5")
        generator.save_weights(generator_checkpoint)
        discriminator.save_weights(discriminator_checkpoint)
        with open(progress_path, "w") as f:
            json.dump({"last_epoch": epoch}, f)

flush_image_buffer()
generator.save_weights(generator_checkpoint)
discriminator.save_weights(discriminator_checkpoint)
with open(progress_path, "w") as f:
    json.dump({"last_epoch": num_epochs - 1}, f)
print("آموزش کامل شد.")


## 6. ذخیره‌ی generator و تولید تصویر جدید

In [ ]:
generator.save("generator_model.h5")

noise = np.random.normal(0, 1, (16, latent_dimensions))
imgs = np.clip(0.5 * generator.predict(noise, verbose=0) + 0.5, 0, 1)

fig, axs = plt.subplots(4, 4, figsize=(6, 6))
for k in range(16):
    axs[k // 4, k % 4].imshow(imgs[k])
    axs[k // 4, k % 4].axis("off")
plt.show()
